[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/04_data_preprocessing/90_exercises.ipynb)

# Bài tập — Chương 04: Data Preprocessing (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/04_data_preprocessing

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_shape, check_true
from ml_course.utils import repo_path

rng = np.random.default_rng(0)
churn = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))

## L0 — Nhớ khái niệm

**Câu 1.** Viết công thức min-max scaling, z-score và mean normalization. Mỗi cách cho kết quả nằm trong khoảng nào / có trung bình bao nhiêu?

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Giải thích vì sao Decision Tree không cần scaling còn KNN thì gần như bắt buộc.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Train: `[10, 20, 30, 40]`. Tính (a) min-max của giá trị test `50`; (b) z-score của `50` (std với ddof=0); (c) robust scale `(x − median)/IQR` của `50` với $Q_1 = 17.5$, $Q_3 = 32.5$.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
minmax_50 = None
z_50 = None
robust_50 = None
### END CODE HERE ###

In [ ]:
check_close(minmax_50, 4 / 3, message="(a) min-max — lớn hơn 1 vì 50 > max của train")
check_close(z_50, 25 / np.sqrt(125), message="(b) z-score")
check_close(robust_50, 25 / 15, message="(c) robust")

## L3 — Code

**Bài 4.** Cài đặt `StandardScalerManual` với `fit`/`transform` (cột hằng → chia cho 1) và `inverse_transform`.

In [ ]:
class StandardScalerManual:
    def fit(self, X):
        X = np.asarray(X, dtype=float)
        ### START CODE HERE ### (≈ 3 dòng)
        pass
        ### END CODE HERE ###
        return self

    def transform(self, X):
        ### START CODE HERE ### (≈ 1 dòng)
        Z = None
        ### END CODE HERE ###
        return Z

    def inverse_transform(self, Z):
        ### START CODE HERE ### (≈ 1 dòng)
        X = None
        ### END CODE HERE ###
        return X

In [ ]:
Xa = np.column_stack([rng.normal(5, 3, 50), np.full(50, 2.0)])
s = StandardScalerManual().fit(Xa)
check_close(s.transform(Xa), StandardScaler().fit_transform(Xa), message="khớp sklearn (kể cả cột hằng)")
check_close(s.inverse_transform(s.transform(Xa)), Xa, message="inverse_transform")

**Bài 5.** Viết `target_encode_oof(cat, y, n_splits=5, m=10, seed=0)`: target encoding **out-of-fold** có smoothing cho mảng category `cat` của tập train. Với mỗi fold, thống kê (số lượng $n_c$, trung bình $\bar y_c$) tính trên các fold còn lại; $\text{enc} = \frac{n_c\bar y_c + m\bar y}{n_c + m}$ với $\bar y$ là trung bình target của các fold còn lại; category chưa gặp → $\bar y$.

In [ ]:
from sklearn.model_selection import KFold


def target_encode_oof(cat, y, n_splits=5, m=10, seed=0):
    cat, y = np.asarray(cat), np.asarray(y, dtype=float)
    enc = np.zeros(len(y))
    for fit_idx, enc_idx in KFold(n_splits, shuffle=True, random_state=seed).split(cat):
        ### START CODE HERE ### (≈ 5 dòng)
        stats_ = None
        prior = None
        smooth = None
        ### END CODE HERE ###
    return enc

In [ ]:
noise = rng.integers(0, 500, 3000).astype(str)
y_rand = rng.integers(0, 2, 3000)
enc = target_encode_oof(noise, y_rand)
from sklearn.metrics import roc_auc_score
check_shape(enc, (3000,))
check_true(abs(roc_auc_score(y_rand, enc) - 0.5) < 0.05, "category ngẫu nhiên → AUC ≈ 0.5 (không leakage)")
city = rng.choice(["A", "B"], 4000)
y_city = (rng.random(4000) < np.where(city == "A", 0.8, 0.2)).astype(int)
enc_city = target_encode_oof(city, y_city)
check_true(enc_city[city == "A"].mean() > 0.7 and enc_city[city == "B"].mean() < 0.3, "bắt được tín hiệu thật")

## L4 — Debug

**Bài 6.** Đoạn code sau cho kết quả CV lạc quan vì **scaler được fit trên toàn bộ dữ liệu** trước khi cross-validate (và tương tự với các bước học từ dữ liệu). Viết lại bằng `Pipeline` để mọi bước nằm trong CV. (Với scaler, chênh lệch nhỏ; với feature selection/oversampling — chương 05, 18, 19 — chênh lệch rất lớn.)

In [ ]:
X = churn[["tenure_months", "monthly_charges", "support_calls", "n_products"]].to_numpy(float)
y = churn["churn"].to_numpy()

# 🐛 cách làm sai
X_scaled_all = StandardScaler().fit_transform(X)
leaky = cross_val_score(LogisticRegression(), X_scaled_all, y, cv=5).mean()

### START CODE HERE ### (≈ 2 dòng)
pipe = None
correct = None
### END CODE HERE ###
print(f"leaky = {leaky:.4f} | đúng = {correct:.4f}")

In [ ]:
check_true(isinstance(pipe, Pipeline) and "StandardScaler" in str(pipe.steps), "dùng Pipeline có StandardScaler")
check_true(0.6 < correct < 0.9, "điểm CV hợp lý")

## L5 — Phân tích

**Bài 7.** So sánh 3 cách mã hoá `contract` (ordinal theo thứ tự `month-to-month < one_year < two_year`, ordinal theo bảng chữ cái, one-hot) cho Logistic Regression dự đoán churn (chỉ dùng `contract` + `tenure_months` chuẩn hoá). Ghi lại accuracy CV và giải thích.

In [ ]:
Xe = churn[["contract", "tenure_months"]]
results = {}
encoders = {
    "ordinal đúng thứ tự": OrdinalEncoder(categories=[["month-to-month", "one_year", "two_year"]]),
    "ordinal bảng chữ cái": OrdinalEncoder(),
    "one-hot": OneHotEncoder(handle_unknown="ignore"),
}
### START CODE HERE ### (≈ 5 dòng)
pass
### END CODE HERE ###
pd.Series(results).round(4)

In [ ]:
check_equal(len(results), 3, "đủ 3 cách mã hoá")
check_true(results["one-hot"] >= results["ordinal bảng chữ cái"] - 1e-3, "one-hot không tệ hơn ordinal sai thứ tự")

✍️ Giải thích:

## L6 — Thực tế

**Bài 8.** Xây pipeline hoàn chỉnh dự đoán churn với **mọi cột** (trừ `customer_id`): cột số (impute median + scale), cột phân loại (impute `"missing"` + one-hot), `senior` passthrough. Model: Logistic Regression (`max_iter=1000`). Báo cáo ROC-AUC 5-fold CV trên tập train (80%) và ROC-AUC trên test (20%, `stratify=y`, `random_state=0`).

In [ ]:
X_all = churn.drop(columns=["customer_id", "churn"])
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y, test_size=0.2, stratify=y, random_state=0)
num_cols = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
cat_cols = ["contract", "internet_service", "payment_method"]
### START CODE HERE ### (≈ 10 dòng)
preprocess = None
churn_model = None
cv_auc = None
test_auc = None
### END CODE HERE ###
print(f"CV AUC = {cv_auc:.4f} | test AUC = {test_auc:.4f}")

In [ ]:
check_greater(cv_auc, 0.75, "CV ROC-AUC > 0.75")
check_true(abs(cv_auc - test_auc) < 0.05, "CV và test nhất quán (không leakage)")

## L7 — Nâng cao

**Bài 9.** Viết transformer `RareCategoryGrouper(min_count=20)` theo chuẩn sklearn: `fit` ghi nhớ, với **mỗi cột**, các category xuất hiện ≥ `min_count` lần trong train; `transform` thay mọi category khác (kể cả category chưa gặp) bằng `"OTHER"`. Dùng được trong `Pipeline` trước `OneHotEncoder`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class RareCategoryGrouper(TransformerMixin, BaseEstimator):
    def __init__(self, min_count=20):
        self.min_count = min_count

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        ### START CODE HERE ### (≈ 2 dòng)
        pass
        ### END CODE HERE ###
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        ### START CODE HERE ### (≈ 2 dòng)
        pass
        ### END CODE HERE ###
        return X

In [ ]:
toy = pd.DataFrame({"city": ["A"] * 30 + ["B"] * 25 + ["C"] * 3 + ["D"] * 2})
g = RareCategoryGrouper(min_count=20).fit(toy)
check_equal(sorted(g.transform(toy)["city"].unique()), ["A", "B", "OTHER"], "gộp C, D thành OTHER")
check_equal(g.transform(pd.DataFrame({"city": ["A", "Z"]}))["city"].tolist(), ["A", "OTHER"], "category lạ → OTHER")
pipe = Pipeline([("rare", RareCategoryGrouper(20)), ("oh", OneHotEncoder(sparse_output=False))]).fit(toy)
check_shape(pipe.transform(toy), (60, 3))